In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

In [6]:
DATA_DIR = Path(
    r"C:\Users\admin\Desktop\HWU\ResearchMethods\MachineLearningCVE"
)

OUTPUT_DIR = DATA_DIR / "processed"

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")

for file in csv_files:
    print(file.name)

Found 8 CSV files:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [7]:
all_data = []

for file in csv_files:
    print(f"Processing: {file.name}")

    df = pd.read_csv(file)

    # Remove spaces around column names
    df.columns = df.columns.str.strip()

    # Keep source file for later analysis
    df["SourceFile"] = file.name

    all_data.append(df)

data = pd.concat(all_data, ignore_index=True)

print("Original shape:", data.shape)

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Processing: Monday-WorkingHours.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Processing: Wednesday-workingHours.pcap_ISCX.csv
Original shape: (2830743, 80)


In [8]:
data = data.drop(columns=["Fwd Header Length.1"])

print("After removing duplicated feature:", data.shape)

After removing duplicated feature: (2830743, 79)


In [9]:
data.replace([np.inf, -np.inf], np.nan, inplace=True)

before_invalid = len(data)

data = data.dropna()

after_invalid = len(data)

print("Rows removed because of NaN/Infinity:",
      before_invalid - after_invalid)

print("Remaining rows:", after_invalid)

Rows removed because of NaN/Infinity: 2867
Remaining rows: 2827876


In [10]:
duplicate_columns = [
    column for column in data.columns
    if column != "SourceFile"
]

global_duplicates = data.duplicated(
    subset=duplicate_columns,
    keep="first"
)

print("Global duplicate rows:", global_duplicates.sum())

Global duplicate rows: 307078


In [11]:
duplicate_rows = data[global_duplicates]

global_duplicates_by_label = (
    duplicate_rows["Label"]
    .value_counts()
)

print(global_duplicates_by_label)

Label
BENIGN                      176263
PortScan                     68110
DoS Hulk                     57278
SSH-Patator                   2678
FTP-Patator                   2004
DoS slowloris                  411
DoS Slowhttptest               271
Web Attack � Brute Force        37
DDoS                            11
Bot                              8
DoS GoldenEye                    7
Name: count, dtype: int64


In [12]:
global_duplicate_summary = []

for label, total in data["Label"].value_counts().items():

    duplicates = global_duplicates_by_label.get(label, 0)

    global_duplicate_summary.append({
        "Label": label,
        "Total_after_invalid_removal": total,
        "Global_Duplicates": duplicates,
        "Duplicate_Percent": duplicates / total * 100
    })

global_duplicate_summary_df = pd.DataFrame(
    global_duplicate_summary
)

global_duplicate_summary_df = (
    global_duplicate_summary_df
    .sort_values("Duplicate_Percent", ascending=False)
    .reset_index(drop=True)
)

global_duplicate_summary_df

,Label,Total_after_invalid_removal,Global_Duplicates,Duplicate_Percent
0,SSH-Patator,5897,2678,45.412922
1,PortScan,158804,68110,42.889348
2,FTP-Patator,7935,2004,25.255198
3,DoS Hulk,230124,57278,24.890059
4,BENIGN,2271320,176263,7.760377
5,DoS slowloris,5796,411,7.091097
6,DoS Slowhttptest,5499,271,4.928169
7,Web Attack � Brute Force,1507,37,2.455209
8,Bot,1956,8,0.408998
9,DoS GoldenEye,10293,7,0.068007


In [13]:
before_duplicates = len(data)

data = data.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
)

after_duplicates = len(data)

print("Rows removed as exact duplicates:",
      before_duplicates - after_duplicates)

print("Rows remaining after duplicate removal:",
      after_duplicates)

Rows removed as exact duplicates: 307078
Rows remaining after duplicate removal: 2520798


In [14]:
data["BinaryLabel"] = (
    data["Label"]
    .astype(str)
    .str.strip()
    .ne("BENIGN")
    .astype(int)
)

In [15]:
print("Binary class counts:")
print(data["BinaryLabel"].value_counts().sort_index())

print("\nBinary class percentages:")
print(
    data["BinaryLabel"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

Binary class counts:
BinaryLabel
0    2095057
1     425741
Name: count, dtype: int64

Binary class percentages:
BinaryLabel
0    83.110864
1    16.889136
Name: proportion, dtype: float64


In [16]:
numeric_cols = data.select_dtypes(include=[np.number]).columns

print("Remaining NaN values:",
      data.isna().sum().sum())

print("Remaining Infinity values:",
      np.isinf(data[numeric_cols]).sum().sum())

print("Remaining exact duplicates:",
      data.duplicated(
          subset=[
              col for col in data.columns
              if col not in ["SourceFile", "BinaryLabel"]
          ]
      ).sum())

Remaining NaN values: 0
Remaining Infinity values: 0
Remaining exact duplicates: 0


In [17]:
OUTPUT_FILE = OUTPUT_DIR / "cicids2017_clean.csv"

data.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Clean dataset saved to:")
print(OUTPUT_FILE)

print("Final shape:", data.shape)

Clean dataset saved to:
C:\Users\admin\Desktop\HWU\ResearchMethods\MachineLearningCVE\processed\cicids2017_clean.csv
Final shape: (2520798, 80)
